In [3]:
###document structure
from langchain_core.documents import Document

In [4]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "Author":"krish naik",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'Author': 'krish naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [5]:
##create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [6]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [7]:
#textloader
from langchain_community.document_loaders import TextLoader
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [8]:
###Directory Loader
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=False
)
documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [9]:
from langchain_community.document_loaders import PyMuPDFLoader
dir_loader=DirectoryLoader(
"../data/pdf",
glob="**/*.pdf",
loader_cls=PyMuPDFLoader,
show_progress=False
)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Wdesk Fidelity Content Translations Version 011.005.072', 'creator': 'Workiva', 'creationdate': '2025-04-25T01:07:35+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'file_path': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 49, 'format': 'PDF 1.4', 'title': 'GOOG 10-Q Q1 2025', 'author': 'anonymous', 'subject': '', 'keywords': '', 'moddate': '2025-04-25T01:07:35+00:00', 'trapped': '', 'modDate': "D:20250425010735+00'00'", 'creationDate': "D:20250425010735+00'00'", 'page': 0}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n________________________________________________________________________________________\nFORM 10-Q \n________________________________________________________________________________________\n(Mark One)\n☒\nQUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the quarterly period ended March 31, 2025\nOR\n☐\nTRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF T

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [11]:
chunks = split_documents(documents)

Split 2 documents into 2 chunks

Example chunk:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}


embedding and VectorStorDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [13]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
           texts:List of text strings to embed
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dim)"""
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager      
    


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 160.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.Embedding dimension: 384


VectorStore

In [14]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory: str="../data/vector_store"):
        """
        Initialize the vector store

        Args:
         collection_name:Name of the chromadb collection
         persist_directory:Directory to persist the vector store 
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector stor initialized.Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error Initializing vector store: {e}")
            raise
    def add_documents(self,documents: List[Any],embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
        documents:List of Langchain documents
        embeddings:corresponding embeddings for the documents"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} document to the vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore

           
                  



Vector stor initialized.Collection: pdf_documents
Existing documents in collection: 2


In [15]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [ ]:
texts=[doc.page_content for doc in chunks]
texts
embeddings=embedding_manager.generate_embeddings(texts)
print(embeddings)
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s]

Generated embeddings with shape: (2, 384)
[[-3.15599367e-02 -3.29403095e-02  4.52735722e-02 -2.80028749e-02
   3.45180407e-02  2.26807874e-02  1.27942627e-02 -6.15102090e-02
  -1.22174978e-01 -1.72238017e-03 -6.71481267e-02  4.71375138e-03
   2.80751977e-02 -3.56527120e-02 -2.84920889e-03  5.31844841e-03
   3.91743518e-02 -1.08010340e-02 -9.46508348e-03 -7.42369667e-02
   4.31737863e-02  1.60430856e-02 -4.06411998e-02  3.04173529e-02
   8.15043878e-03  5.22306412e-02  6.50505498e-02  4.15119603e-02
   5.27264588e-02 -2.00100765e-02 -9.94715025e-04 -4.57338989e-02
   5.20965783e-04  5.74598275e-02 -6.97539151e-02 -1.86431650e-02
  -2.34254412e-02  1.54415919e-02 -1.08244885e-02 -5.80926193e-03
  -4.65660878e-02 -4.29417714e-02  1.43903019e-02 -1.15238745e-02
   1.52299896e-01  9.40929279e-02 -9.92160290e-02 -1.05190240e-01
  -2.30506472e-02 -1.76740866e-02 -5.79000190e-02 -4.36255671e-02
   4.11176262e-03  7.45747006e-03 -8.25635120e-02  2.15802994e-02
   3.48044671e-02  8.33706930e-02 

In [17]:
class RAGretriever:
    def __init__(self,vector_store: VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int=5,score_threshold: float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        Args:
         query:The search query
         top_k:number of top results to return
         score_threshold:minimum similarity score
        Returns:
        list of dictionaries containing retrieved documents and metadata """
        print(f"Reteieving documents for query: '{query}'")
        print(f"Top k : {top_k}, Score threshold: {score_threshold}")
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distanced=results['distances'][0]
                ids=results['ids'][0]
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distanced)):
                    similarity_score=1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_sore':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retrieval=RAGretriever(vectorstore,embedding_manager)


In [18]:
rag_retrieval

In [19]:
rag_retrieval.retrieve("what is Machine Learning")

Reteieving documents for query: 'what is Machine Learning'
Top k : 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_fcc924c1_0',
  'content': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems',
  'metadata': {'content_length': 568,
   'doc_index': 0,
   'source': '..\\data\\text_files\\machine_learning.txt'},
  'similarity_sore': 0.502154529094696,
  'distance': 0.49784547090530396,
  'rank': 1},
 {'id': 'doc_491fd1ad_0',
  'content': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom e